# What's the deal with all the injuries??

In [3]:
import pandas as pd
import plotly.express as px

In [ ]:
master_prem_fixtures = pd.read_csv('processed data\master_premier_league_fixtures.csv')
master_absences = pd.read_csv('processed data\master_absence_long.csv')

<>:3: SyntaxWarning: invalid escape sequence '\m'
<>:4: SyntaxWarning: invalid escape sequence '\m'
<>:3: SyntaxWarning: invalid escape sequence '\m'
<>:4: SyntaxWarning: invalid escape sequence '\m'
C:\Users\GARETHE\AppData\Local\Temp\ipykernel_28764\525279041.py:3: SyntaxWarning: invalid escape sequence '\m'
  master_prem_fixtures = pd.read_csv('processed data\master_premier_league_fixtures.csv')
C:\Users\GARETHE\AppData\Local\Temp\ipykernel_28764\525279041.py:4: SyntaxWarning: invalid escape sequence '\m'
  master_absences = pd.read_csv('processed data\master_absence_long.csv')


### Is there correlation between total games played by a team and the average number of gameweeks missed by a player in the prem?

In [6]:
# (Assuming master_prem_fixtures and master_absences are already loaded in memory)

# ==========================================
# 1. CALCULATE GAMES PLAYED PER TEAM/SEASON
# ==========================================
games_played = master_prem_fixtures.groupby(['team_name', 'season']).size().reset_index(name='games_played')

# ==========================================
# 2. CALCULATE SQUAD SIZE PER TEAM/SEASON
# ==========================================
# We count the unique number of players registered to each team per season
squad_sizes = master_absences.groupby(['team_name', 'season'])['player_name'].nunique().reset_index(name='squad_size')

# ==========================================
# 3. CALCULATE TOTAL INJURIES
# ==========================================
injuries_df = master_absences[master_absences['player_status'] == 'Absence/injury']
injury_counts = injuries_df.groupby(['team_name', 'season']).size().reset_index(name='total_injuries')

# ==========================================
# 4. MERGE AND CALCULATE THE AVERAGE
# ==========================================
# Merge the base metrics together
plot_df = pd.merge(games_played, squad_sizes, on=['team_name', 'season'], how='left')
plot_df = pd.merge(plot_df, injury_counts, on=['team_name', 'season'], how='left')

# Fill NaNs for teams with miraculously 0 injuries
plot_df['total_injuries'] = plot_df['total_injuries'].fillna(0)

# Calculate the critical metric: Average gameweeks missed per player
plot_df['avg_injuries_per_player'] = plot_df['total_injuries'] / plot_df['squad_size']

# Round to 2 decimal places for cleaner hover tooltips
plot_df['avg_injuries_per_player'] = plot_df['avg_injuries_per_player'].round(2)

# ==========================================
# 5. GENERATE PLOTLY GRAPH
# ==========================================
fig = px.scatter(
    plot_df,
    x='games_played',
    y='avg_injuries_per_player',
    color='team_name',          
    hover_data=['season', 'squad_size', 'total_injuries'], # Added deeper context to hover
    trendline='ols',            
    trendline_scope='overall',  
    title='Impact of Match Congestion on Average Player Injury Rates (Top 6 Premier League)',
    labels={
        'games_played': 'Total Matches Played (All Competitions)',
        'avg_injuries_per_player': 'Average Gameweeks Missed per Player',
        'team_name': 'Club',
        'squad_size': 'Squad Size',
        'total_injuries': 'Total Injury Days'
    }
)

# Polish the visuals
fig.update_traces(marker=dict(size=12, opacity=0.8, line=dict(width=1, color='DarkSlateGrey')))
fig.update_layout(template='plotly_white', title_x=0.5)

fig.show()

### Has average gameweeks missed per player increased over time?

In [ ]:
import pandas as pd
from prophet import Prophet
from prophet.plot import plot_plotly
import plotly.express as px

# (Assuming master_absences is already loaded in memory)

# ==========================================
# 1. CALCULATE OVERALL AVERAGES PER SEASON
# ==========================================
# Get total squad sizes per season across all Top 6 teams
team_squads = master_absences.groupby(['season', 'team_name'])['player_name'].nunique().reset_index(name='squad_size')
season_squads = team_squads.groupby('season')['squad_size'].sum().reset_index()

# Get total injuries per season across all Top 6 teams
injuries_df = master_absences[master_absences['player_status'] == 'Absence/injury']
season_injuries = injuries_df.groupby('season').size().reset_index(name='total_injuries')

# Merge and calculate the average
ts_df = pd.merge(season_squads, season_injuries, on='season', how='left')
ts_df['total_injuries'] = ts_df['total_injuries'].fillna(0)
ts_df['avg_injuries'] = ts_df['total_injuries'] / ts_df['squad_size']


# ==========================================
# 2. FORMAT DATA FOR PROPHET
# ==========================================
# Prophet strictly requires columns named 'ds' (datestamp) and 'y' (metric)
# We convert "2019-2020" -> "2019-08-01" (Approximate start of the season)
ts_df['ds'] = pd.to_datetime(ts_df['season'].str[:4] + '-08-01')
ts_df['y'] = ts_df['avg_injuries']

prophet_df = ts_df[['ds', 'y']].copy()

print("[*] Training Prophet Model on historical injury data...")

# ==========================================
# 3. TRAIN PROPHET & FORECAST
# ==========================================
# Initialize model (disabling smaller seasonalities since our data is yearly)
m = Prophet(yearly_seasonality=False, weekly_seasonality=False, daily_seasonality=False)
m.fit(prophet_df)

# Create future timestamps for the next 3 seasons (2026, 2027, 2028)
future_years = [2026, 2027, 2028]
future_dates = pd.to_datetime([f"{y}-08-01" for y in future_years])
future_df = pd.DataFrame({'ds': pd.concat([prophet_df['ds'], pd.Series(future_dates)], ignore_index=True)})

# Predict the future trend
forecast = m.predict(future_df)

# ==========================================
# 4. PLOT WITH PLOTLY
# ==========================================
# Use Prophet's native Plotly integration for beautiful interactive graphs
fig = plot_plotly(m, forecast)

fig.update_layout(
    title='Prophet Forecast: Average Gameweeks Missed per Player (Top 6 Premier League)',
    title_x=0.5,
    xaxis_title='Season Start Date',
    yaxis_title='Average Gameweeks Missed',
    template='plotly_white',
    hovermode='x unified'
)

fig.show()

[*] Training Prophet Model on historical injury data...


23:02:50 - cmdstanpy - INFO - Chain [1] start processing
23:02:51 - cmdstanpy - INFO - Chain [1] done processing


#### How does this trend compare to number of games played for a team over time?

In [8]:
import pandas as pd
from prophet import Prophet
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ==========================================
# 1. DATA AGGREGATION
# ==========================================
# A. Average Injuries per Player (From previous logic)
team_squads = master_absences.groupby(['season', 'team_name'])['player_name'].nunique().reset_index(name='squad_size')
season_squads = team_squads.groupby('season')['squad_size'].sum().reset_index()

injuries_df = master_absences[master_absences['player_status'] == 'Absence/injury']
season_injuries = injuries_df.groupby('season').size().reset_index(name='total_injuries')

ts_injuries = pd.merge(season_squads, season_injuries, on='season', how='left')
ts_injuries['total_injuries'] = ts_injuries['total_injuries'].fillna(0)
ts_injuries['avg_injuries'] = ts_injuries['total_injuries'] / ts_injuries['squad_size']

# B. Average Games Played per Top 6 Team
games_df = master_prem_fixtures.groupby(['season', 'team_name']).size().reset_index(name='games_played')
season_games = games_df.groupby('season')['games_played'].mean().reset_index(name='avg_games')

# C. Master Time Series Dataset
ts_master = pd.merge(ts_injuries, season_games, on='season', how='inner')
ts_master['ds'] = pd.to_datetime(ts_master['season'].str[:4] + '-08-01')

print("[*] Data aggregated. Training Dual Prophet Models...")

# ==========================================
# 2. TRAIN DUAL PROPHET MODELS
# ==========================================
# Future dates for forecasting (Next 3 seasons)
future_dates = pd.to_datetime([f"{y}-08-01" for y in [2026, 2027, 2028]])
future_df = pd.DataFrame({'ds': pd.concat([ts_master['ds'], pd.Series(future_dates)], ignore_index=True)})

# Model 1: Injuries
m_inj = Prophet(yearly_seasonality=False, weekly_seasonality=False, daily_seasonality=False)
m_inj.fit(ts_master[['ds', 'avg_injuries']].rename(columns={'avg_injuries': 'y'}))
forecast_inj = m_inj.predict(future_df)

# Model 2: Games Played
m_games = Prophet(yearly_seasonality=False, weekly_seasonality=False, daily_seasonality=False)
m_games.fit(ts_master[['ds', 'avg_games']].rename(columns={'avg_games': 'y'}))
forecast_games = m_games.predict(future_df)

# ==========================================
# 3. BUILD DUAL-AXIS PLOTLY GRAPH
# ==========================================
# Create figure with secondary y-axis
fig = make_subplots(specs=[[{"secondary_y": True}]])

# --- TRACE 1: INJURIES (Left Axis - Red) ---
# Actual Data Points
fig.add_trace(
    go.Scatter(x=ts_master['ds'], y=ts_master['avg_injuries'], name="Actual Avg Injuries",
               mode='markers', marker=dict(color='red', size=10)),
    secondary_y=False,
)
# Prophet Trendline
fig.add_trace(
    go.Scatter(x=forecast_inj['ds'], y=forecast_inj['trend'], name="Trend: Injuries",
               mode='lines', line=dict(color='red', dash='dot')),
    secondary_y=False,
)

# --- TRACE 2: GAMES PLAYED (Right Axis - Blue) ---
# Actual Data Points
fig.add_trace(
    go.Scatter(x=ts_master['ds'], y=ts_master['avg_games'], name="Actual Avg Games",
               mode='markers', marker=dict(color='blue', size=10)),
    secondary_y=True,
)
# Prophet Trendline
fig.add_trace(
    go.Scatter(x=forecast_games['ds'], y=forecast_games['trend'], name="Trend: Games Played",
               mode='lines', line=dict(color='blue', dash='dot')),
    secondary_y=True,
)

# ==========================================
# 4. POLISH AND DISPLAY
# ==========================================
fig.update_layout(
    title='Prophet Forecast: Match Congestion vs. Injury Rates Over Time',
    title_x=0.5,
    template='plotly_white',
    hovermode='x unified',
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

# Axis Titles
fig.update_xaxes(title_text="Season Start Date")
fig.update_yaxes(title_text="<b>Avg Gameweeks Missed</b> (Per Player)", color="red", secondary_y=False)
fig.update_yaxes(title_text="<b>Avg Matches Played</b> (Per Team)", color="blue", secondary_y=True)

fig.show()

[*] Data aggregated. Training Dual Prophet Models...


23:07:25 - cmdstanpy - INFO - Chain [1] start processing
23:07:25 - cmdstanpy - INFO - Chain [1] done processing
23:07:25 - cmdstanpy - INFO - Chain [1] start processing
23:07:26 - cmdstanpy - INFO - Chain [1] done processing


The above plot is counterintuitive in that the number of matches played by the top 6 clubs (on average) doesn't have an upward trend that matches the avg GW missed per player. This says a few things:

- Top 6 clubs are not playing more matches on average every year
- Number of club matches played by the top 6 doesn't correlate to more gws missed by injury per player

So what is causing the increased injuries?
- Either we are seeing more international matches over time?
- Match tactics are high press and too demanding.

More international matches cannot be the issue if we do not already see a clear correlation between more matches played and more gws missed per player.

#### How does this trend compare to average match density over time?

In [10]:
import pandas as pd
from prophet import Prophet
from prophet.plot import plot_plotly

# (Assuming master_prem_fixtures is already loaded in memory)

# ==========================================
# 1. CALCULATE REST DAYS PER MATCH
# ==========================================
# Ensure date column is a datetime object
master_prem_fixtures['date'] = pd.to_datetime(master_prem_fixtures['date'])

# Sort chronologically per team to calculate the exact rest between games
fixtures_sorted = master_prem_fixtures.sort_values(by=['team_name', 'date'])

# Calculate difference in days between consecutive matches
fixtures_sorted['rest_days'] = fixtures_sorted.groupby(['team_name', 'season'])['date'].diff().dt.days

# ==========================================
# 2. AGGREGATE TO A SEASONAL LEAGUE AVERAGE
# ==========================================
# Drop the first match of the season (which has NaN rest days)
valid_rest_days = fixtures_sorted.dropna(subset=['rest_days'])

# Calculate the overall average rest days across ALL Top 6 teams per season
season_rest_df = valid_rest_days.groupby('season')['rest_days'].mean().reset_index(name='avg_rest_days')

# ==========================================
# 3. FORMAT FOR PROPHET
# ==========================================
# Convert "2019-2020" string into a datestamp ("2019-08-01") for Prophet
season_rest_df['ds'] = pd.to_datetime(season_rest_df['season'].str[:4] + '-08-01')
season_rest_df['y'] = season_rest_df['avg_rest_days']

prophet_df = season_rest_df[['ds', 'y']].copy()

print("[*] Training Prophet Model on historical schedule density...")

# ==========================================
# 4. TRAIN PROPHET & FORECAST
# ==========================================
# Initialize model (disabling sub-yearly seasonalities since our data is aggregated per year)
m = Prophet(yearly_seasonality=False, weekly_seasonality=False, daily_seasonality=False)
m.fit(prophet_df)

# Create future timestamps for the next 3 seasons (2026, 2027, 2028)
future_years = [2026, 2027, 2028]
future_dates = pd.to_datetime([f"{y}-08-01" for y in future_years])
future_df = pd.DataFrame({'ds': pd.concat([prophet_df['ds'], pd.Series(future_dates)], ignore_index=True)})

# Predict the future trend
forecast = m.predict(future_df)

# ==========================================
# 5. PLOT WITH PLOTLY
# ==========================================
fig = plot_plotly(m, forecast)

fig.update_layout(
    title='Prophet Forecast: Average Rest Days Between Matches (Top 6 Premier League)',
    title_x=0.5,
    xaxis_title='Season Start Date',
    yaxis_title='Average Rest Days Between Fixtures',
    template='plotly_white',
    hovermode='x unified'
)

fig.show()

[*] Training Prophet Model on historical schedule density...


23:26:34 - cmdstanpy - INFO - Chain [1] start processing
23:26:34 - cmdstanpy - INFO - Chain [1] done processing


In [ ]:
import pandas as pd
from prophet import Prophet
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# (Assuming master_prem_fixtures and master_absences are already loaded in memory)

# ==========================================
# 1. DATA AGGREGATION
# ==========================================
# A. Average Injuries per Player 
team_squads = master_absences.groupby(['season', 'team_name'])['player_name'].nunique().reset_index(name='squad_size')
season_squads = team_squads.groupby('season')['squad_size'].sum().reset_index()

injuries_df = master_absences[master_absences['player_status'] == 'Absence/injury']
season_injuries = injuries_df.groupby('season').size().reset_index(name='total_injuries')

ts_injuries = pd.merge(season_squads, season_injuries, on='season', how='left')
ts_injuries['total_injuries'] = ts_injuries['total_injuries'].fillna(0)
ts_injuries['avg_injuries'] = ts_injuries['total_injuries'] / ts_injuries['squad_size']

# B. Average Rest Days per Match
master_prem_fixtures['date'] = pd.to_datetime(master_prem_fixtures['date'])
fixtures_sorted = master_prem_fixtures.sort_values(by=['team_name', 'date'])
fixtures_sorted['rest_days'] = fixtures_sorted.groupby(['team_name', 'season'])['date'].diff().dt.days
valid_rest_days = fixtures_sorted.dropna(subset=['rest_days'])
season_rest_df = valid_rest_days.groupby('season')['rest_days'].mean().reset_index(name='avg_rest_days')

# C. Master Time Series Dataset
ts_master = pd.merge(ts_injuries, season_rest_df, on='season', how='inner')
ts_master['ds'] = pd.to_datetime(ts_master['season'].str[:4] + '-08-01')

# ==========================================
# 2. CALCULATE CORRELATION & R-SQUARED
# ==========================================
# Calculate Pearson Correlation Coefficient (r)
correlation = ts_master['avg_rest_days'].corr(ts_master['avg_injuries'])

# Calculate R-squared
r_squared = correlation ** 2

print(f"[*] Pearson Correlation (r): {correlation:.3f}")
print(f"[*] R-squared (R²): {r_squared:.3f}")

# ==========================================
# 3. TRAIN DUAL PROPHET MODELS
# ==========================================
future_dates = pd.to_datetime([f"{y}-08-01" for y in [2026, 2027, 2028]])
future_df = pd.DataFrame({'ds': pd.concat([ts_master['ds'], pd.Series(future_dates)], ignore_index=True)})

# Model 1: Injuries
m_inj = Prophet(yearly_seasonality=False, weekly_seasonality=False, daily_seasonality=False)
m_inj.fit(ts_master[['ds', 'avg_injuries']].rename(columns={'avg_injuries': 'y'}))
forecast_inj = m_inj.predict(future_df)

# Model 2: Rest Days
m_rest = Prophet(yearly_seasonality=False, weekly_seasonality=False, daily_seasonality=False)
m_rest.fit(ts_master[['ds', 'avg_rest_days']].rename(columns={'avg_rest_days': 'y'}))
forecast_rest = m_rest.predict(future_df)

# ==========================================
# 4. BUILD DUAL-AXIS PLOTLY GRAPH
# ==========================================
fig = make_subplots(specs=[[{"secondary_y": True}]])

# --- TRACE 1: INJURIES (Left Axis - Red) ---
fig.add_trace(
    go.Scatter(x=ts_master['ds'], y=ts_master['avg_injuries'], name="Actual Avg Injuries",
               mode='markers', marker=dict(color='red', size=10)),
    secondary_y=False,
)
fig.add_trace(
    go.Scatter(x=forecast_inj['ds'], y=forecast_inj['trend'], name="Trend: Injuries",
               mode='lines', line=dict(color='red', dash='dot')),
    secondary_y=False,
)

# --- TRACE 2: REST DAYS (Right Axis - Green) ---
fig.add_trace(
    go.Scatter(x=ts_master['ds'], y=ts_master['avg_rest_days'], name="Actual Avg Rest Days",
               mode='markers', marker=dict(color='green', size=10)),
    secondary_y=True,
)
fig.add_trace(
    go.Scatter(x=forecast_rest['ds'], y=forecast_rest['trend'], name="Trend: Rest Days",
               mode='lines', line=dict(color='green', dash='dot')),
    secondary_y=True,
)

# ==========================================
# 5. POLISH AND ANNOTATE
# ==========================================
fig.update_layout(
    title='Prophet Forecast: Match Congestion (Rest Days) vs. Injury Rates Over Time',
    title_x=0.5,
    template='plotly_white',
    hovermode='x unified',
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

fig.update_xaxes(title_text="Season Start Date")
fig.update_yaxes(title_text="<b>Avg Gameweeks Missed</b> (Per Player)", color="red", secondary_y=False)
fig.update_yaxes(title_text="<b>Avg Rest Days Between Matches</b>", color="green", secondary_y=True, autorange="reversed")

fig.show()

[*] Pearson Correlation (r): 0.054
[*] R-squared (R²): 0.003


23:32:22 - cmdstanpy - INFO - Chain [1] start processing
23:32:22 - cmdstanpy - INFO - Chain [1] done processing
23:32:23 - cmdstanpy - INFO - Chain [1] start processing
23:32:23 - cmdstanpy - INFO - Chain [1] done processing


So this plot tells us that the trend lines are correlating, but the points are not well correlated.

Football is becoming more match dense for the top 6 clubs AND there are more games missed in the prem per player on average but it does not mean that more game density leads to more gameweeks missed.

### Has number of distinct player injuries increased over time?

In [14]:
import pandas as pd
import plotly.express as px

# (Assuming master_absences is already loaded in memory)

# ==========================================
# 1. DATA AGGREGATION: DISTINCT PLAYERS
# ==========================================
# Filter to only look at actual injuries
injuries_df = master_absences[master_absences['player_status'] == 'Absence/injury']

# Count the unique number of individual players injured per team, per season
distinct_injured = injuries_df.groupby(['team_name', 'season'])['player_name'].nunique().reset_index(name='unique_injured_players')

# Create a Datetime column so the OLS trendline math works properly
distinct_injured['ds'] = pd.to_datetime(distinct_injured['season'].str[:4] + '-08-01')

# ==========================================
# 2. GENERATE CLUB-LEVEL TREND GRAPH
# ==========================================
# By putting 'team_name' in the color parameter, Plotly automatically draws a separate trendline for each club!
fig = px.scatter(
    distinct_injured,
    x='ds',
    y='unique_injured_players',
    color='team_name',
    trendline='ols', 
    title='Spread of the Crisis: Distinct Players Injured per Season (by Club)',
    labels={
        'ds': 'Season',
        'unique_injured_players': 'Total Unique Players Injured',
        'team_name': 'Club'
    }
)

# Polish visuals
fig.update_traces(marker=dict(size=12, opacity=0.8, line=dict(width=1, color='DarkSlateGrey')))
fig.update_layout(template='plotly_white', title_x=0.5)

# Format the X-axis to show the readable '2019-2020' string instead of the raw date
unique_dates = sorted(distinct_injured['ds'].unique())
unique_seasons = sorted(distinct_injured['season'].unique())
fig.update_xaxes(tickvals=unique_dates, ticktext=unique_seasons)

fig.show()

### How strongly does average gameweeks missed per player influence whether a team wins the league or not

In [15]:
import pandas as pd
import plotly.express as px

# (Assuming master_absences is loaded)

# ==========================================
# 1. CALCULATE AVG INJURIES PER PLAYER
# ==========================================
squad_sizes = master_absences.groupby(['team_name', 'season'])['player_name'].nunique().reset_index(name='squad_size')

injuries_df = master_absences[master_absences['player_status'] == 'Absence/injury']
injury_counts = injuries_df.groupby(['team_name', 'season']).size().reset_index(name='total_injuries')

inj_stats = pd.merge(squad_sizes, injury_counts, on=['team_name', 'season'], how='left')
inj_stats['total_injuries'] = inj_stats['total_injuries'].fillna(0)
inj_stats['avg_injuries'] = (inj_stats['total_injuries'] / inj_stats['squad_size']).round(2)

# ==========================================
# 2. INJECT ACTUAL LEAGUE FINISHING POSITIONS
# ==========================================
# Historical league positions for the Top 6 teams (19/20 through 23/24)
# Note: 24/25 is excluded as the season is not finished yet.
league_positions = {
    '2019-2020': {'Liverpool': 1, 'Man City': 2, 'Man United': 3, 'Chelsea': 4, 'Tottenham': 6, 'Arsenal': 8},
    '2020-2021': {'Man City': 1, 'Man United': 2, 'Liverpool': 3, 'Chelsea': 4, 'Tottenham': 7, 'Arsenal': 8},
    '2021-2022': {'Man City': 1, 'Liverpool': 2, 'Chelsea': 3, 'Tottenham': 4, 'Arsenal': 5, 'Man United': 6},
    '2022-2023': {'Man City': 1, 'Arsenal': 2, 'Man United': 3, 'Liverpool': 5, 'Tottenham': 8, 'Chelsea': 12},
    '2023-2024': {'Man City': 1, 'Arsenal': 2, 'Liverpool': 3, 'Tottenham': 5, 'Chelsea': 6, 'Man United': 8},
    '2024-2025': {'Liverpool': 1, 'Arsenal': 2, 'Man City': 3, 'Chelsea': 4, 'Man United': 15, 'Tottenham': 17}
}

# Flatten the dictionary into a DataFrame for merging
pos_data = []
for season, teams in league_positions.items():
    for team, rank in teams.items():
        # Standardize the team name to match our Transfermarkt formatting
        tm_team_name = team.replace(' ', '_').title() if 'Man' in team else team
        if tm_team_name == 'Man_United': tm_team_name = 'Man United'
        if tm_team_name == 'Man_City': tm_team_name = 'Man City'
        
        pos_data.append({'season': season, 'team_name': tm_team_name, 'league_rank': rank})

rank_df = pd.DataFrame(pos_data)

# ==========================================
# 3. MERGE DATASETS
# ==========================================
# We use an inner join so we only plot completed seasons (dropping 2024-2025)
plot_df = pd.merge(inj_stats, rank_df, on=['team_name', 'season'], how='inner')

# Calculate the correlation for an annotation
correlation = plot_df['avg_injuries'].corr(plot_df['league_rank'])

# ==========================================
# 4. GENERATE PLOTLY GRAPH
# ==========================================
fig = px.scatter(
    plot_df,
    x='avg_injuries',
    y='league_rank',
    color='team_name',
    hover_data=['season'],
    trendline='ols', 
    trendline_scope='overall',
    title='Does Squad Health Predict League Success? (Top 6: 2019 - 2024)',
    labels={
        'avg_injuries': 'Avg Gameweeks Missed (Per Player)',
        'league_rank': 'Final League Position',
        'team_name': 'Club'
    }
)

# --- CRITICAL VISUAL TWEAK ---
# In sports, Rank #1 is the "Top". By reversing the Y-axis, the top of the graph 
# represents winning the league, making the visual intuitive to read.
fig.update_yaxes(autorange="reversed", tickmode='linear', dtick=1)

# Polish the visuals
fig.update_traces(marker=dict(size=14, opacity=0.85, line=dict(width=1, color='DarkSlateGrey')))
fig.update_layout(template='plotly_white', title_x=0.5)

# Add Correlation text box
stat_text = f"<b>Correlation (r):</b> {correlation:.2f}<br><i>(Positive = More injuries correlate with dropping lower down the table)</i>"
fig.add_annotation(
    x=0.98, y=0.95, xref="paper", yref="paper",
    text=stat_text, showarrow=False,
    font=dict(size=13, color="black"), align="right",
    bgcolor="rgba(255,255,255,0.9)", bordercolor="black", borderwidth=1, borderpad=8
)

fig.show()

A Pearson correlation of 0.36 is a moderate positive correlation.
In the complex world of football—where referee decisions, tactical genius, billion-pound budgets, and pure luck all influence the final table—the fact that we can mathematically tie a noticeable chunk of league success purely to how empty the medical room is, is highly significant.